## Libraries

In [25]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langsmith import Client
from langchain.smith import RunEvalConfig, run_on_dataset
from langsmith.utils import LangSmithConflictError

## Data Ingestion and LLM runner

In [26]:
class DatasetIngestor:
    def __init__(self, dataset_name, qa_examples, description="Factual QA"):
        self.client = Client()
        self.dataset_name = dataset_name
        self.qa_examples = qa_examples
        self.description = description

    def reset_and_ingest(self):
        # Delete dataset if exists
        try:
            dataset = self.client.read_dataset(dataset_name=self.dataset_name)
            self.client.delete_dataset(dataset_id=dataset.id)
            print(f"Deleted existing dataset: {self.dataset_name}")
        except Exception:
            pass
        # Create new dataset
        dataset = self.client.create_dataset(dataset_name=self.dataset_name, description=self.description)
        print(f"Created dataset: {self.dataset_name}")
        # Add examples
        for ex in self.qa_examples:
            self.client.create_example(inputs=ex["inputs"], outputs=ex["outputs"], dataset_id=dataset.id)
        print(f"Added {len(self.qa_examples)} examples.")

class LLMRunEvaluator:
    def __init__(self, dataset_name, project_name, llm=None):
        self.client = Client(
            api_url=os.environ["LANGSMITH_ENDPOINT"],
            api_key=os.environ["LANGSMITH_API_KEY"],
        )
        self.dataset_name = dataset_name
        self.project_name = project_name
        self.llm = llm or ChatOpenAI(model="gpt-4", temperature=0)
        self.chain = self._build_chain()

    def _build_chain(self):
        prompt = ChatPromptTemplate.from_messages([
            ("system", "Answer the question accurately and clearly."),
            ("human", "{question}")
        ])
        return prompt | self.llm | StrOutputParser()


    def run_evaluation(self):
        # Create or reuse project session
        try:
            project = self.client.create_project(
                project_name=self.project_name,
                description="QA evaluation project",
                upsert=True
            )
            print(f"Project session ready: {project.name}")
        except LangSmithConflictError:
            print("Reusing existing project session.")
            project = self.client.read_project(project_name=self.project_name)
            print(f"Project session ready: {project.name}")

        evaluation = RunEvalConfig(
            evaluators=["qa"],
            reference_key="answer",
            prediction_key="output",
            eval_llm=self.llm
        )

        results = run_on_dataset(
            client=self.client,
            dataset_name=self.dataset_name,
            llm_or_chain_factory=lambda: self.chain,
            evaluation=evaluation,
            verbose=True
        )
        print("✅ Evaluation complete! Check your results at your LangSmith dashboard.")

        # --- Enrich: Print analytics and failed runs summary ---
        # Fetch all runs for this project
        try:
            runs = list(self.client.list_runs(project_name=self.project_name, execution_order=1))
            total = len(runs)
            failed = [r for r in runs if r.error]
            print(f"\nLangSmith Analytics Summary:")
            print(f"Total runs: {total}")
            print(f"Failed runs: {len(failed)}")
            if failed:
                print("\nFailed run details:")
                for r in failed:
                    print(f"- Run ID: {r.id}, Error: {r.error}")
            # Print a few sample results
            print("\nSample results:")
            for r in runs[:3]:
                print(f"Input: {r.inputs}")
                print(f"Output: {r.outputs}")
                print(f"Reference: {r.reference_outputs if hasattr(r, 'reference_outputs') else 'N/A'}")
                print(f"Score: {r.evaluation_results if hasattr(r, 'evaluation_results') else 'N/A'}\n")
        except Exception as e:
            print(f"Could not fetch analytics: {e}")


## Run the Code

In [27]:

# qa_examples = [
#     {"inputs": {"question": "What is the capital of Germany?"}, "outputs": {"answer": "Berlin"}},
#     {"inputs": {"question": "Who wrote Hamlet?"}, "outputs": {"answer": "William Shakespeare"}},
#     {"inputs": {"question": "Speed of light in vacuum?"}, "outputs": {"answer": "299,792,458 m/s"}}
# ]
dataset_name = "Trivia QA"
project_name = "pr-large-ladybug-30"

# Step 1: Ingest data (reset dataset and add examples)
# ingestor = DatasetIngestor(dataset_name, qa_examples)
# ingestor.reset_and_ingest()

# Step 2: Evaluate LLM on dataset
evaluator = LLMRunEvaluator(dataset_name, project_name)
evaluator.run_evaluation() 

Reusing existing project session.
Project session ready: pr-large-ladybug-30
View the evaluation results for project 'long-van-95' at:
https://smith.langchain.com/o/0e211d37-f90d-4c1f-93d9-01c61642acd4/datasets/cfe170f5-889f-46bd-a546-d94eca69ee9e/compare?selectedSessions=49b53100-8be5-4b14-8189-7fd3a5807bfb

View all tests for Dataset Trivia QA at:
https://smith.langchain.com/o/0e211d37-f90d-4c1f-93d9-01c61642acd4/datasets/cfe170f5-889f-46bd-a546-d94eca69ee9e
[------------------------------------------------->] 3/3

,feedback.correctness,error,execution_time,run_id
count,3.0,0,3.000000,3
unique,NaN,0,NaN,3
top,NaN,NaN,NaN,9ca026fc-09c0-4475-8a2e-a9abcfa448a8
freq,NaN,NaN,NaN,1
mean,1.0,NaN,0.859967,NaN
std,0.0,NaN,0.306214,NaN
min,1.0,NaN,0.666133,NaN
25%,1.0,NaN,0.683458,NaN
50%,1.0,NaN,0.700783,NaN
75%,1.0,NaN,0.956884,NaN


✅ Evaluation complete! Check your results at your LangSmith dashboard.

LangSmith Analytics Summary:
Total runs: 0
Failed runs: 0

Sample results:
